# Question 7 — Exercise A.7

**V** = walk camera ↔ ride last wagon. Also WASD / QE.



In [ ]:
%%html

<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecshader.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecapp.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/cgraobject.js"></script>
<script src="https://git.io/glm-js.min.js"></script>


<script id="my-vertex-shader" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
attribute vec3 in_Color;
uniform mat4 MVP;
varying vec3 ex_Color;
void main(void){
  gl_Position = MVP * vec4(in_Position.x, in_Position.y, in_Position.z, 1.0);
  ex_Color = in_Color;
}
</script>


<script id="my-fragment-shader" type="x-shader/x-fragment">
precision mediump float;
varying vec3 ex_Color;
void main(void){ gl_FragColor = vec4(ex_Color, 1.0); }
</script>


<div id="cgraStatus" style="font:13px/1.4 system-ui,sans-serif;color:#0a7;min-height:1.2em;"></div>
<script>
function cgraReady(){
  return typeof DEECapp!=='undefined' && typeof DEECshader!=='undefined'
      && typeof CGRAobject!=='undefined' && typeof glm!=='undefined';
}
function cgraBoot(fn){
  var n=0;
  (function wait(){
    n++;
    if(!cgraReady()){
      if(n===30){ var e=document.getElementById('cgraStatus'); if(e){ e.style.color='#c60'; e.textContent='Waiting libraries...'; } }
      if(n<250) return setTimeout(wait,40);
      var e2=document.getElementById('cgraStatus'); if(e2){ e2.style.color='#c00'; e2.textContent='ERROR: libs not loaded (F12)'; }
      return;
    }
    var e3=document.getElementById('cgraStatus'); if(e3) e3.textContent='';
    try{ fn(); }catch(err){ console.error(err); alert('Error: '+err.message); }
  })();
}
</script>

<div style="font:14px/1.45 system-ui,sans-serif;color:#243;margin:0 0 8px 0;max-width:800px;">
  <b>Q7 Toggle</b> — key <b>v</b> ride wagon<br>
  <span style="color:#666;">Voilà only (not Run). Click canvas before keys.</span>
</div>
<canvas id="cQ7" width="800" height="520"
  style="border:1px solid #2a3340;border-radius:8px;background:#1a2230;display:block;box-shadow:0 8px 24px rgba(0,0,0,.18);"></canvas>
<script>
cgraBoot(function(){
// ============================================================
// CGRA A1 — BASIC shapes (all extend CGRAobject)
// Keep this simple for oral defense.
// Color: smooth gradient from color1 to color2 along X.
// ============================================================

function mixColor(c1, c2, t) {
    // t from 0 to 1
    return [
        c1[0] * (1 - t) + c2[0] * t,
        c1[1] * (1 - t) + c2[1] * t,
        c1[2] * (1 - t) + c2[2] * t
    ];
}

function pushTri(verts, cols, ax, ay, az, bx, by, bz, cx, cy, cz, c1, c2) {
    // one triangle = 3 vertices
    var pts = [[ax, ay, az], [bx, by, bz], [cx, cy, cz]];
    for (var i = 0; i < 3; i++) {
        verts.push(pts[i][0], pts[i][1], pts[i][2]);
        // gradient by X: -1..1 mapped to 0..1
        var t = (pts[i][0] + 1.0) * 0.5;
        if (t < 0) t = 0;
        if (t > 1) t = 1;
        var col = mixColor(c1, c2, t);
        cols.push(col[0], col[1], col[2]);
    }
}

function uploadBuffers(obj, verts, cols) {
    obj.numvertices = verts.length / 3;
    obj.vertexbuffer = obj.gl.createBuffer();
    obj.gl.bindBuffer(obj.gl.ARRAY_BUFFER, obj.vertexbuffer);
    obj.gl.bufferData(obj.gl.ARRAY_BUFFER, new Float32Array(verts), obj.gl.STATIC_DRAW);

    obj.colorbuffer = obj.gl.createBuffer();
    obj.gl.bindBuffer(obj.gl.ARRAY_BUFFER, obj.colorbuffer);
    obj.gl.bufferData(obj.gl.ARRAY_BUFFER, new Float32Array(cols), obj.gl.STATIC_DRAW);
}

// ---------- CUBE: 6 faces * 2 triangles * 3 verts = 36 ----------
class Cube extends CGRAobject {
    constructor(glcontext, color1, color2) {
        super(glcontext);
        var c1 = color1 || [1, 0, 0];
        var c2 = color2 || [0, 0, 1];
        var verts = [];
        var cols = [];

        // unit cube corners from -1 to 1
        // faces: front, back, left, right, top, bottom
        // each face = 2 triangles
        var f = [
            // front z=+1
            [-1, -1, 1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, 1, 1, -1, 1, 1],
            // back z=-1
            [1, -1, -1, -1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, 1, -1],
            // left x=-1
            [-1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -1, -1, -1, 1, 1, -1, 1, -1],
            // right x=+1
            [1, -1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1, 1, 1, -1, 1, 1, 1],
            // top y=+1
            [-1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, 1, 1, -1, -1, 1, -1],
            // bottom y=-1
            [-1, -1, -1, 1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, -1, -1, 1]
        ];

        for (var fi = 0; fi < f.length; fi++) {
            var a = f[fi];
            // two triangles already listed as 6 verts
            pushTri(verts, cols, a[0], a[1], a[2], a[3], a[4], a[5], a[6], a[7], a[8], c1, c2);
            pushTri(verts, cols, a[9], a[10], a[11], a[12], a[13], a[14], a[15], a[16], a[17], c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- DISC: center + ring (>=10) ----------
class Disc extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [1, 1, 0];
        var c2 = color2 || [1, 0.5, 0];
        n = n || 12; // at least 10
        var verts = [];
        var cols = [];
        var r = 1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // triangle: center, p0, p1  (in XZ plane, y=0)
            pushTri(verts, cols, 0, 0, 0, x0, 0, z0, x1, 0, z1, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- CONE: apex + base circle (>=10) ----------
class Cone extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [0.8, 0.4, 0.1];
        var c2 = color2 || [0.3, 0.15, 0.05];
        n = n || 12;
        var verts = [];
        var cols = [];
        var r = 1.0;
        var apexY = 1.0;
        var baseY = -1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // side triangle
            pushTri(verts, cols, 0, apexY, 0, x0, baseY, z0, x1, baseY, z1, c1, c2);
            // base triangle (optional close)
            pushTri(verts, cols, 0, baseY, 0, x1, baseY, z1, x0, baseY, z0, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- CYLINDER: two circles (>=10), quads -> 2 tris ----------
class Cylinder extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [0.6, 0.6, 0.6];
        var c2 = color2 || [0.2, 0.2, 0.2];
        n = n || 12;
        var verts = [];
        var cols = [];
        var r = 1.0;
        var yTop = 1.0, yBot = -1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // side: two triangles
            pushTri(verts, cols, x0, yBot, z0, x1, yBot, z1, x1, yTop, z1, c1, c2);
            pushTri(verts, cols, x0, yBot, z0, x1, yTop, z1, x0, yTop, z0, c1, c2);
            // top cap
            pushTri(verts, cols, 0, yTop, 0, x0, yTop, z0, x1, yTop, z1, c1, c2);
            // bottom cap
            pushTri(verts, cols, 0, yBot, 0, x1, yBot, z1, x0, yBot, z0, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- SPHERE: latitude strips (>=10) * rings (>=10) ----------
class Sphere extends CGRAobject {
    constructor(glcontext, color1, color2, stacks, slices) {
        super(glcontext);
        var c1 = color1 || [0.2, 0.6, 1.0];
        var c2 = color2 || [1.0, 0.2, 0.4];
        stacks = stacks || 12; // latitude strips
        slices = slices || 12; // vertices per ring
        var verts = [];
        var cols = [];
        var R = 1.0;

        for (var i = 0; i < stacks; i++) {
            // elevation from -PI/2 to +PI/2
            var lat0 = -Math.PI / 2 + (i / stacks) * Math.PI;
            var lat1 = -Math.PI / 2 + ((i + 1) / stacks) * Math.PI;
            var y0 = R * Math.sin(lat0);
            var y1 = R * Math.sin(lat1);
            var r0 = R * Math.cos(lat0);
            var r1 = R * Math.cos(lat1);

            for (var j = 0; j < slices; j++) {
                var a0 = (j / slices) * 2 * Math.PI;
                var a1 = ((j + 1) / slices) * 2 * Math.PI;

                var x00 = r0 * Math.cos(a0), z00 = r0 * Math.sin(a0);
                var x01 = r0 * Math.cos(a1), z01 = r0 * Math.sin(a1);
                var x10 = r1 * Math.cos(a0), z10 = r1 * Math.sin(a0);
                var x11 = r1 * Math.cos(a1), z11 = r1 * Math.sin(a1);

                // two triangles between rings
                pushTri(verts, cols, x00, y0, z00, x01, y0, z01, x11, y1, z11, c1, c2);
                pushTri(verts, cols, x00, y0, z00, x11, y1, z11, x10, y1, z10, c1, c2);
            }
        }
        uploadBuffers(this, verts, cols);
    }
}

function T(x,y,z){ return glm.translate(glm.mat4(1.0), glm.vec3(x,y,z)); }
function Rdeg(deg,x,y,z){ return glm.rotate(glm.mat4(1.0), glm.radians(deg), glm.vec3(x,y,z)); }
function Su(s){ return glm.mat4(glm.mat3(s)); }
function mul2(a,b){ return a['*'](b); }
function mul3(a,b,c){ return a['*'](b['*'](c)); }
function mul4(a,b,c,d){ return a['*'](b['*'](c['*'](d))); }
function makeShader(app){
  var fs=document.getElementById('my-fragment-shader').text;
  var vs=document.getElementById('my-vertex-shader').text;
  app.shaderprog=new DEECshader(app.gl);
  app.shaderprog.srcShaders(vs,fs);
  app.gl.enable(app.gl.DEPTH_TEST);
}
function setSh(obj, app){ obj.setShader(app.shaderprog); }
function draw(obj, M){ obj.setModelTransformation(M); obj.drawit(); }
function drawVP(obj, M, view, proj){ obj.setModelTransformation(M); obj.drawit(view, proj); }

function lemni(w,a,b){
  var s=Math.sin(w), c=Math.cos(w), d=1+s*s;
  return {x:a*c/d, z:b*s*c/d};
}
function lemniD(w,a,b,eps){
  var s=Math.sin(w), c=Math.cos(w), d=1+s*s, d2=d*d;
  return { dx:-a*eps*s*(2+c*c)/d2, dz:eps*b*(Math.cos(2*w)*d - s*c*2*s*c)/d2 };
}
class myapp extends DEECapp{
  camMode=0;
  eye={x:0,y:0.9,z:3.2}; yaw=0; keys={};
  initialize(){
    makeShader(this);
    this.gl.clearColor(0.52, 0.70, 0.90, 1);
    this.grass=new Cube(this.gl,[0.34,0.58,0.30],[0.18,0.40,0.18]);
    this.wood=new Cube(this.gl,[0.76,0.48,0.22],[0.40,0.20,0.08]);
    this.stone=new Cube(this.gl,[0.64,0.66,0.72],[0.36,0.38,0.44]);
    this.ram=new Cone(this.gl,[1.0,0.78,0.28],[0.80,0.35,0.10],12);
    this.roof=new Cone(this.gl,[0.85,0.28,0.25],[0.45,0.10,0.10],12);
    var a=[this.grass,this.wood,this.stone,this.ram,this.roof];
    for(var i=0;i<a.length;i++) setSh(a[i],this);
    this.a=1.6; this.b=1.05; this.eps=0.48; this.theta=[0,-0.5,-1.0,-1.5];
    this.t0=Date.now();
    window.addEventListener('keydown',(e)=>{
      var k=e.key.toLowerCase(); this.keys[k]=true;
      if(k==='v'){ this.camMode=1-this.camMode; console.log(this.camMode?'ride':'walk'); }
    },false);
    window.addEventListener('keyup',(e)=>{ this.keys[e.key.toLowerCase()]=false; },false);
  }
  render(){
    this.gl.clear(this.gl.COLOR_BUFFER_BIT|this.gl.DEPTH_BUFFER_BIT);
    var t=(Date.now()-this.t0)*0.001;
    var wagons=[];
    for(var i=0;i<4;i++){
      var w=this.eps*t+this.theta[i];
      var p=lemni(w,this.a,this.b);
      var d=lemniD(w,this.a,this.b,this.eps);
      wagons.push({x:p.x, z:p.z, yaw:Math.atan2(d.dx,d.dz)});
    }
    var aspect=this.canvas.width/this.canvas.height;
    var proj=glm.perspective(glm.radians(55),aspect,0.05,80);
    var view;
    if(this.camMode===0){
      var sp=0.045, rad=this.yaw*Math.PI/180;
      var fx=Math.sin(rad), fz=-Math.cos(rad);
      var rx=Math.cos(rad), rz=Math.sin(rad);
      if(this.keys['w']){ this.eye.x+=fx*sp; this.eye.z+=fz*sp; }
      if(this.keys['s']){ this.eye.x-=fx*sp; this.eye.z-=fz*sp; }
      if(this.keys['a']){ this.eye.x-=rx*sp; this.eye.z-=rz*sp; }
      if(this.keys['d']){ this.eye.x+=rx*sp; this.eye.z+=rz*sp; }
      if(this.keys['q']) this.yaw-=1.5;
      if(this.keys['e']) this.yaw+=1.5;
      view=glm.lookAt(glm.vec3(this.eye.x,this.eye.y,this.eye.z),
                      glm.vec3(this.eye.x+fx,this.eye.y,this.eye.z+fz), glm.vec3(0,1,0));
    } else {
      var W=wagons[3];
      var fx=Math.sin(W.yaw), fz=Math.cos(W.yaw);
      view=glm.lookAt(glm.vec3(W.x-fx*0.35,0.55,W.z-fz*0.35),
                      glm.vec3(W.x+fx*2.2,0.4,W.z+fz*2.2), glm.vec3(0,1,0));
    }
    function D(obj,M){ drawVP(obj,M,view,proj); }
    D(this.grass, mul2(T(0,-0.35,0), Su(3.2)));
    D(this.stone, mul2(T(0,0.4,-5.2), Su(1.6)));
    D(this.roof, mul2(T(-1.2,1.5,-5.0), Su(0.55)));
    D(this.roof, mul2(T( 1.2,1.5,-5.0), Su(0.55)));
    for(var i=0;i<4;i++){
      var W=wagons[i];
      var M=mul3(T(W.x,0.22,W.z), Rdeg(W.yaw*180/Math.PI,0,1,0), Su(0.22));
      D(this.wood, M);
      if(i===0) D(this.ram, mul4(T(W.x,0.28,W.z), Rdeg(W.yaw*180/Math.PI,0,1,0), T(0,0,0.42), Su(0.16)));
    }
  }
}
new myapp('cQ7').run();

});
</script>

